In [ ]:
from pathlib import Path
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "project.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside this repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project import input_path, output_path
import pandas as pd
df = pd.read_csv(input_path('analysis'))
dfField = pd.read_csv(input_path('fields'))
df['Patent'] = (df['Patent_Count'] > 0).astype(float).where(df['Patent_Count'].notna())
df['Tweet'] = (df['Tweet_Count'] > 0).astype(float).where(df['Tweet_Count'].notna())


In [ ]:
#Author Alex J. Yang, alexjieyang@outlook.com
import numpy as np
import pandas as pd
import itertools
import scipy.stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from tqdm import tqdm

# c

In [ ]:
df1 = df[df['Patent']>0].copy()
df2 = df[df['Tweet']>0].copy()
df3 = df[(df['Patent']==0) & (df['Tweet']==0)].copy()

In [ ]:
x_pos = [1,2,3,]
colors = ["#C0392B","#AF601A","#212F3D",]

In [ ]:
d = [len(df1) , len(df2) , len(df3)]

f, ax = plt.subplots(figsize=(10, 9))
plt.bar(x_pos, d,align='center', color= colors,alpha=0.9,capsize=10)
ax.yaxis.grid(True)
ax.set_yscale("log")
plt.xticks(x_pos,['Cited by\nPatents' , 'Cited by\nTweets','Not Cited by\nPatents/Tweets' , ],fontsize = 28)
for _ in range(len(d)):
    if d[_]>1000000:
        plt.text(x_pos[_],d[_]*0.55 , str(d[_]//1000000)+'M',fontsize = 25,c = 'white', ha='center', fontweight='bold')
    else:
        plt.text(x_pos[_],d[_]*0.55 , str(d[_]//1000)+'K',fontsize = 25,c = 'white', ha='center', fontweight='bold')
plt.yticks([10e3,10e4,10e5,10e6,10e7],size = 28)
plt.ylabel(r'Number',fontsize = 28)
plt.savefig(output_path('fig SM 2/panel_1.svg'), bbox_inches='tight')


# d

In [ ]:
year = range(2000,2016)
y0 = df ['C5'].groupby(df['Year']).count().reindex(year, fill_value=0).values
y1 = df1['C5'].groupby(df1['Year']).count().reindex(year, fill_value=0).values
y2 = df2['C5'].groupby(df2['Year']).count().reindex(year, fill_value=0).values
y3 = df3['C5'].groupby(df3['Year']).count().reindex(year, fill_value=0).values

In [ ]:
markers = ['o','^','D']
labels = ['Cited by\nPatents' , 'Cited by\nTweets','Not Cited by\nPatents/Tweets' , ]

In [ ]:
f, ax = plt.subplots(figsize=(20, 9))
for i in range(3):
    y = [y1,y2,y3][i]
    plt.plot(year , y , marker = markers[i], markersize = 15, label=labels[i], color=colors[i])
ax.set_yscale("log")
plt.legend(fontsize = 25 , loc = 'lower left')
plt.xticks([2000,2005,2010,2015],size = 28)
plt.yticks([10,100,1000,10000,100000,1000000],size = 28)
plt.ylabel(r'Number',fontsize = 28)
plt.savefig(output_path('fig SM 2/panel_2.svg'), bbox_inches='tight')


In [ ]:
year = np.array(range(2000,2016))
y10 = y1/y0
y20 = y2/y0
f, ax = plt.subplots(figsize=(11, 6))
plt.bar(year-.05 , y10 ,linewidth = 2.5,fill = False, edgecolor=colors[0],)
plt.bar(year+.05 , y20 ,linewidth = 2.5,fill = False, edgecolor=colors[1],)
# plt.legend(fontsize = 25 , loc = 'lower right')
plt.xticks([2000,2005,2010,2015],size = 28)
plt.yticks(size = 28)
plt.ylabel(r'Proportion',fontsize = 28)
plt.savefig(output_path('fig SM 2/panel_3.svg'), bbox_inches='tight')


# a

In [ ]:
Fieldlst = dfField['FieldID'].values

In [ ]:
data1 = []
data2 = []
year = range(2000,2016)
for Field in Fieldlst:
    df0 = df.loc[df['FieldID'] == Field,:].copy()
    y0 = df0['Patent'].groupby(df0['Year']).count().reindex(year, fill_value=0).values
    y1 = df0['Patent'].groupby(df0['Year']).sum().reindex(year, fill_value=0).values
    y2 = df0['Tweet'].groupby(df0['Year']).sum().reindex(year, fill_value=0).values
    y10 = y1/y0
    y20 = y2/y0
    data1.append(y10)
    data2.append(y20)

In [ ]:
dd1 = pd.DataFrame(data1 , columns=year , index = dfField['Field_Name'].values)
dd2 = pd.DataFrame(data2 , columns=year , index = dfField['Field_Name'].values)
dd1 = dd1*100
dd2 = dd2*100

In [ ]:
plt.subplots(1,2,figsize=(22,14))
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0,hspace=0)
sns.set_theme(style="ticks")

ax = plt.subplot(1,2, 1)
ax.set_title("Proportion of Papers Cited by Patents (%)", fontsize=28)
cmap = sns.color_palette('Reds', as_cmap=True)
heatmap = sns.heatmap(data=dd1, cmap=cmap, vmin=0, vmax=15,
                      square=True, linewidths=1.5, cbar_kws={"shrink": .9,"location":"bottom",'fraction':.05,"pad":.03,})
ax.tick_params(labelsize=25)
heatmap.set_xticks([])
# Add data labels
for i in range(len(dd1)):
    for j in range(len(dd1.columns)):
        if dd1.iloc[i, j]<3:
            text = ax.text(j + 0.5, i + 0.5, round(dd1.iloc[i, j], 1),
                           ha="center", va="center", color="black", fontsize=18)
        else:
            text = ax.text(j + 0.5, i + 0.5, round(dd1.iloc[i, j], 1),
                           ha="center", va="center", color="white", fontsize=18, fontweight='bold')


ax = plt.subplot(1,2, 2)
ax.set_title("Proportion of Papers Cited by Tweets (%)", fontsize=28)
cmap = sns.color_palette('Oranges', as_cmap=True)
heatmap = sns.heatmap(data=dd2, cmap=cmap, vmin=0, vmax=8,
                      square=True, linewidths=1.5, cbar_kws={"shrink": .9,"location":"bottom",'fraction':.05,"pad":.03,})
ax.tick_params(labelsize=25)
heatmap.set_yticks([])
heatmap.set_xticks([])
# Add data labels
for i in range(len(dd2)):
    for j in range(len(dd2.columns)):
        if dd2.iloc[i, j]<2:
            text = ax.text(j + 0.5, i + 0.5, round(dd2.iloc[i, j], 1),
                           ha="center", va="center", color="black", fontsize=18)
        else:
            text = ax.text(j + 0.5, i + 0.5, round(dd2.iloc[i, j], 1),
                           ha="center", va="center", color="white", fontsize=18, fontweight='bold')
plt.savefig(output_path('fig SM 2/panel_4.svg'), bbox_inches='tight')
